# 02 — Fine-Tuning (`gemma-3-270m-it` LoRA SFT on Cleaned Dataset `v2`)

Trains a LoRA adapter in FP32 for **3 epochs** (`cosine` schedule, `lr=2e-4`, `warmup_ratio=0.08`, `weight_decay=0.01`, effective batch `8`, `gradient_checkpointing=False` for 4x T4 speedup within `8.4 GiB` peak VRAM) using assistant-only loss (`labels == -100` on prompts) and `FiniteLossTrainer`.

### 1. Install Dependencies
Installs the pinned Hugging Face, PEFT, Accelerate, and dataset packages required for FP32 LoRA training.

In [1]:
%pip install -q "transformers==4.57.6" "datasets==4.8.5" "peft==0.18.1" "accelerate==1.15.0" pandas huggingface_hub nbclient nbformat


### 2. Load Verified `v2` Dataset & Contract
Verifies `train` (`1,325`) and `validation` (`30`) split SHA-256 hashes against the baseline contract and configures training hyperparameters.

In [2]:
import os, json, hashlib, random, re, time, shutil, importlib.metadata
from pathlib import Path
from getpass import getpass
from collections import Counter
import pandas as pd
import torch
from datasets import load_from_disk
from huggingface_hub import get_token
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    LogitsProcessor,
    LogitsProcessorList,
)

def digest(value):
    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True).encode()).hexdigest()

def file_sha(path):
    sha = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            sha.update(block)
    return sha.hexdigest()

def read_token():
    token = os.environ.get("HF_TOKEN") or get_token()
    if not token:
        token = getpass("Hugging Face read token: ").strip()
    if not token:
        raise RuntimeError("A Hugging Face token with accepted Gemma access is required.")
    os.environ["HF_TOKEN"] = token
    return token

def runtime_path(path, sentinel):
    path = Path(path).expanduser()
    candidates = [path]
    if not path.is_absolute():
        candidates += [parent / path for parent in Path.cwd().parents]
        candidates.append(Path("/content") / path)
    for candidate in candidates:
        if (candidate / sentinel).is_file():
            return candidate.resolve()
    raise FileNotFoundError(f"Kernel directory: {Path.cwd()}; missing {sentinel} under {path}.")

class GeneratedOnlyRepetitionPenalty(LogitsProcessor):
    """Applies standard repetition penalty only to generated post-prompt tokens."""
    def __init__(self, penalty: float, prompt_length: int):
        if penalty <= 0:
            raise ValueError("penalty must be > 0")
        self.penalty = float(penalty)
        self.prompt_length = int(prompt_length)

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        if self.penalty == 1.0 or input_ids.shape[1] <= self.prompt_length:
            return scores
        gen_ids = input_ids[:, self.prompt_length :]
        if gen_ids.shape[0] == 1:
            gen_ids = torch.unique(gen_ids[0]).unsqueeze(0)
        score = torch.gather(scores, 1, gen_ids)
        score = torch.where(score < 0, score * self.penalty, score / self.penalty)
        return scores.scatter(1, gen_ids, score)

from transformers import Trainer, TrainingArguments, set_seed
from peft import LoraConfig, TaskType, get_peft_model

PREPARED_ROOT = runtime_path("prepared_clinical_dataset", "latest_v2_5000.json")
BASELINE_DIR = runtime_path("results_v2_5000/baseline", "comparison_contract.json")
contract = json.loads((BASELINE_DIR / "comparison_contract.json").read_text(encoding="utf-8"))
DATASET_VERSION = contract["dataset_version"]
prepared_dir = PREPARED_ROOT / DATASET_VERSION
manifest = json.loads((prepared_dir / "manifest.json").read_text(encoding="utf-8"))
prepared = load_from_disk(str(prepared_dir / "dataset"))
for split, sha in manifest["split_sha256"].items():
    assert digest(prepared[split].to_list()) == sha
assert manifest["split_sha256"]["train"] == contract["train_sha256"]
assert manifest["split_sha256"]["validation"] == contract["validation_sha256"]
assert [c["case_id"] for c in prepared["validation"]] == contract["case_ids"]
assert digest(list(prepared["validation"]["prompt_text"])) == contract["prompt_sha256"]

RUN_NAME = "finetuned_v2_5000"
NUM_EPOCHS = 3
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.08
WEIGHT_DECAY = 0.01
GRADIENT_CHECKPOINTING = False  # Recheck the actual new longest row below.

OUTPUT_DIR = Path("results_v2_5000") / RUN_NAME
if (OUTPUT_DIR / "adapter" / "adapter_config.json").is_file():
    raise FileExistsError(f"{OUTPUT_DIR} already contains a finished adapter.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "comparison_contract.json").write_text(json.dumps(contract, indent=2), encoding="utf-8")

token = read_token()
set_seed(contract["seed"])
tokenizer = AutoTokenizer.from_pretrained(contract["model"], revision=contract["base_model_revision"], token=token)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Verified dataset {DATASET_VERSION}: train={len(prepared['train'])}, val={len(prepared['validation'])}", flush=True)

tokenizer_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Verified dataset 3f8f10951130896b: train=5000, val=100


### 3. Label Mask Checks, LoRA Setup & VRAM Sanity Check
Validates assistant-only label masking (`-100` on prompts), attaches FP32 LoRA (`r=16`, `alpha=32`) to all attention/MLP projections, and runs a forward/backward memory check on the longest (`1,651`-token) row.

In [3]:
def pad_rows(features, pad_id):
    width = max(len(row["input_ids"]) for row in features)
    batch = {key: [] for key in ("input_ids", "attention_mask", "labels")}
    for row in features:
        padding = width - len(row["input_ids"])
        batch["input_ids"].append(list(row["input_ids"]) + [pad_id] * padding)
        batch["attention_mask"].append(list(row["attention_mask"]) + [0] * padding)
        batch["labels"].append(list(row["labels"]) + [-100] * padding)
    return batch

def collate(features):
    return {key: torch.tensor(value, dtype=torch.long) for key, value in pad_rows(features, tokenizer.pad_token_id).items()}

label_checks = {}
for split in ("train", "validation"):
    rows = prepared[split]
    min_sup, max_len = 10**9, 0
    for row in rows:
        ids, mask, labels = row["input_ids"], row["attention_mask"], row["labels"]
        assert len(ids) == len(mask) == len(labels) and len(ids) <= manifest["config"]["max_sequence_length"]
        first_ans = next(i for i, v in enumerate(labels) if v != -100)
        assert labels[:first_ans] == [-100] * first_ans and labels[first_ans:] == ids[first_ans:]
        min_sup = min(min_sup, len(ids) - first_ans)
        max_len = max(max_len, len(ids))
    label_checks[split] = {"rows": len(rows), "min_supervised_tokens": min_sup, "max_total_tokens": max_len}
(OUTPUT_DIR / "label_checks.json").write_text(json.dumps(label_checks, indent=2), encoding="utf-8")
print("Label & mask checks passed:", label_checks, flush=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "T4 GPU runtime is required for training."
base_model = AutoModelForCausalLM.from_pretrained(
    contract["model"], revision=contract["base_model_revision"], token=token, dtype=torch.float32
).to(device)
base_model.config.use_cache = False
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(base_model, lora_config)
if GRADIENT_CHECKPOINTING:
    model.enable_input_require_grads()
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.print_trainable_parameters()

longest_idx = max(range(len(prepared["train"])), key=lambda i: prepared["train"][i]["total_tokens"])
longest_row = prepared["train"][longest_idx]
batch = {k: v.to(device) for k, v in collate([longest_row]).items()}
torch.cuda.reset_peak_memory_stats()
model.train()
model.zero_grad(set_to_none=True)
def backward_check():
    loss = model(**batch).loss
    assert torch.isfinite(loss), "Non-finite initial loss"
    loss.backward()
    grads = [p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
    assert grads and all(bool(torch.isfinite(g).all()) for g in grads), "Non-finite gradients"
    return loss, grads

try:
    loss, grads = backward_check()
except torch.cuda.OutOfMemoryError:
    model.zero_grad(set_to_none=True)
    torch.cuda.empty_cache()
    GRADIENT_CHECKPOINTING = True
    model.enable_input_require_grads()
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    torch.cuda.reset_peak_memory_stats()
    loss, grads = backward_check()
model_check = {
    "longest_case_id": longest_row["case_id"],
    "total_tokens": longest_row["total_tokens"],
    "initial_loss": float(loss.item()),
    "finite_gradients": True,
    "peak_allocated_gib": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    "gpu": torch.cuda.get_device_name(0),
}
model.zero_grad(set_to_none=True)
del batch, grads, loss
torch.cuda.empty_cache()
(OUTPUT_DIR / "model_checks.json").write_text(json.dumps(model_check, indent=2), encoding="utf-8")
print("Backward & VRAM check passed:", model_check, flush=True)

Label & mask checks passed: {'train': {'rows': 5000, 'min_supervised_tokens': 54, 'max_total_tokens': 1651}, 'validation': {'rows': 100, 'min_supervised_tokens': 245, 'max_total_tokens': 1355}}


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

trainable params: 3,796,992 || all params: 271,895,168 || trainable%: 1.3965


Backward & VRAM check passed: {'longest_case_id': 'train:64', 'total_tokens': 1651, 'initial_loss': 1.7577239274978638, 'finite_gradients': True, 'peak_allocated_gib': 8.323, 'gpu': 'Tesla T4'}


### 4. Three-Epoch Training, Checkpoint Selection & Archive
Trains for 3 epochs (`498` optimizer steps) with `FiniteLossTrainer`, selects the lowest-`eval_loss` checkpoint (`checkpoint-498`), saves logs/adapter, and archives `results_v2_5000/finetuned_v2_5000.zip`.

In [4]:
steps_per_epoch = (len(prepared["train"]) + GRADIENT_ACCUMULATION_STEPS - 1) // GRADIENT_ACCUMULATION_STEPS
run_config = {
    "dataset_version": DATASET_VERSION,
    "comparison_contract_sha256": digest(contract),
    "train_sha256": manifest["split_sha256"]["train"],
    "train_examples": len(prepared["train"]),
    "validation_examples": len(prepared["validation"]),
    "model": contract["model"],
    "base_model_revision": contract["base_model_revision"],
    "seed": contract["seed"],
    "num_train_epochs": NUM_EPOCHS,
    "expected_optimizer_steps": steps_per_epoch * NUM_EPOCHS,
    "batch_size": 1,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": WEIGHT_DECAY,
    "max_grad_norm": 1.0,
    "gradient_checkpointing": GRADIENT_CHECKPOINTING,
    "dtype": "float32",
    "lora_rank": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "versions": {p: importlib.metadata.version(p) for p in ("torch", "transformers", "datasets", "peft", "accelerate")},
}
(OUTPUT_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2), encoding="utf-8")

set_seed(contract["seed"])
columns = ["input_ids", "attention_mask", "labels"]
train_data = prepared["train"].select_columns(columns)
eval_data = prepared["validation"].select_columns(columns)

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=1.0,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    bf16=False,
    fp16=False,
    logging_steps=10,
    logging_nan_inf_filter=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    report_to="none",
    seed=contract["seed"],
    data_seed=contract["seed"],
    label_names=["labels"],
    remove_unused_columns=False,
)

class FiniteLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        result = super().compute_loss(model, inputs, return_outputs=return_outputs, **kwargs)
        loss = result[0] if return_outputs else result
        if not torch.isfinite(loss).all():
            raise RuntimeError("Non-finite loss detected; aborting training.")
        return result

from transformers import TrainerCallback
class SaveLiveLog(TrainerCallback):
    def on_log(self, args, state, control, **kwargs):
        pending = OUTPUT_DIR / "training_log.pending"
        pending.write_text(json.dumps(state.log_history, indent=2), encoding="utf-8")
        pending.replace(OUTPUT_DIR / "training_log.json")

trainer = FiniteLossTrainer(
    model=model, args=args, data_collator=collate, train_dataset=train_data, eval_dataset=eval_data, callbacks=[SaveLiveLog()]
)
from transformers.trainer_utils import get_last_checkpoint
last_checkpoint = get_last_checkpoint(str(OUTPUT_DIR / "checkpoints")) if (OUTPUT_DIR / "checkpoints").exists() else None
result = trainer.train(resume_from_checkpoint=last_checkpoint)
assert trainer.state.global_step == 1875 and trainer.state.epoch == 3.0
final_eval = trainer.evaluate()
metrics = {
    "train": result.metrics,
    "best_model_checkpoint": trainer.state.best_model_checkpoint,
    "best_eval_loss": float(trainer.state.best_metric) if trainer.state.best_metric is not None else None,
    "final_eval": final_eval,
}
trainer.save_model(str(OUTPUT_DIR / "adapter"))
tokenizer.save_pretrained(OUTPUT_DIR / "adapter")
(OUTPUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
(OUTPUT_DIR / "training_log.json").write_text(json.dumps(trainer.state.log_history, indent=2), encoding="utf-8")

# Clean temporary optimizer/checkpoint folders after best adapter is saved
if (OUTPUT_DIR / "checkpoints").exists():
    print("Retaining resumable checkpoints until local archival is verified.")

epoch_evals = [e for e in trainer.state.log_history if "eval_loss" in e]
print("Epoch eval losses:", [(round(e.get("epoch", 0), 2), round(e["eval_loss"], 4)) for e in epoch_evals], flush=True)
print("Saved best adapter:", (OUTPUT_DIR / "adapter").resolve(), flush=True)
archive = shutil.make_archive(str(OUTPUT_DIR.resolve()), "zip", root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name)
print("Archive saved:", archive, flush=True)

Epoch,Training Loss,Validation Loss
2,0.453400,0.461478
3,0.415100,0.455239


Retaining resumable checkpoints until local archival is verified.
Epoch eval losses: [(1.0, 0.4979), (2.0, 0.4615), (3.0, 0.4552), (3.0, 0.4552)]


Saved best adapter: /content/clinical_v2_5000/results_v2_5000/finetuned_v2_5000/adapter


Archive saved: /content/clinical_v2_5000/results_v2_5000/finetuned_v2_5000.zip
